In [1]:


jours = [
    ("Soleil", 65, "Faible", "Oui"),  # J1
    ("Soleil", 90, "Faible", "Non"),  # J2
    ("Pluie",  70, "Faible", "Oui"),  # J3
    ("Soleil", 85, "Faible", "Non"),  # J4
    ("Pluie",  80, "Fort",   "Non"),  # J5
    ("Soleil", 70, "Fort",   "Non"),  # J6
    ("Pluie",  60, "Faible", "Oui"),  # J7
    ("Soleil", 75, "Faible", "Oui"),  # J8
]
noms_variables = ["Ciel", "Humidité", "Vent"]
jours = [(c, "Normale" if h <= 77.5 else "Haute", v, j) for (c, h, v, j) in jours]


#  Compter les classes d'un groupe de jours
def compter(groupe):
    comptes = {}
    for jour in groupe:   #répète le bloc pour chaque jour
        classe = jour[3]   #la réponse (Oui/Non)
        comptes[classe] = comptes.get(classe, 0) + 1
    return comptes


# Indice de Gini : 0 = groupe pur, 0,5 = moitié-moitié (pour 2 classes)
def gini(groupe):
    n = len(groupe)  #donne le nombre de jours
    resultat = 1
    for nb in compter(groupe).values():
        resultat -= (nb / n) ** 2
    return resultat


#  Séparer un groupe selon les valeurs de la variable j
def separer(groupe, j):
    sous_groupes = {}
    for jour in groupe:
        v = jour[j]
        if v not in sous_groupes:
            sous_groupes[v] = []
        sous_groupes[v].append(jour)
    return sous_groupes


#  Gain = Gini avant − moyenne pondérée des Gini après
def gain(groupe, j):
    apres = 0
    for sous_groupe in separer(groupe, j).values():
        apres += len(sous_groupe) / len(groupe) * gini(sous_groupe)
    return gini(groupe) - apres


#  ENTRAÎNEMENT : construire l'arbre récursivement
def construire(groupe, variables, profondeur=0, details=True):
    comptes = compter(groupe)
    majorite = max(comptes, key=comptes.get)  #prendre la classe maajoritaire en OUI ou NON
    marge = "    " * profondeur

    # critère d'arrêt : groupe pur ou plus de variable à tester → feuille
    if len(comptes) == 1 or len(variables) == 0:
        return {"feuille": majorite}

    # tester toutes les variables, garder celle au plus grand gain
    meilleure = None
    meilleur_gain = 0
    for j in variables:
        g = gain(groupe, j)
        if details:
            print(f"{marge}gain({noms_variables[j]}) = {g:.3f}")
        if g > meilleur_gain:
            meilleur_gain = g
            meilleure = j

    if meilleure is None:          # aucune question n'améliore → feuille
        return {"feuille": majorite}

    if details:
        print(f"{marge}→ on choisit {noms_variables[meilleure]}  (Gini {gini(groupe):.3f}, {comptes})")

    restantes = [j for j in variables if j != meilleure]
    branches = {}
    for v, sous_groupe in separer(groupe, meilleure).items():
        if details:
            print(f"{marge}  branche {noms_variables[meilleure]} = {v} : {compter(sous_groupe)}")
        branches[v] = construire(sous_groupe, restantes, profondeur + 1, details)
    return {"variable": meilleure, "branches": branches, "majorite": majorite}


#  PRÉDICTION : descendre l'arbre jusqu'à une feuille
def predire(arbre, x):
    while "feuille" not in arbre:
        v = x[arbre["variable"]]
        if v not in arbre["branches"]:     # valeur jamais vue à ce nœud
            return arbre["majorite"]
        arbre = arbre["branches"][v]
    return arbre["feuille"]


#  Afficher l'arbre sous forme de règles
def afficher(arbre, marge=""):
    if "feuille" in arbre:
        print(f"{marge}→ On joue : {arbre['feuille']}")
        return
    for v, sous_arbre in arbre["branches"].items():
        print(f"{marge}SI {noms_variables[arbre['variable']]} = {v} :")
        afficher(sous_arbre, marge + "    ")


# ÉVALUATION : leave-one-out, comme pour KNN
def evaluer(donnees):
    bonnes = 0
    for i in range(len(donnees)):
        test = donnees[i]
        entrainement = donnees[:i] + donnees[i + 1:]
        arbre = construire(entrainement, [0, 1, 2], details=False)
        predite = predire(arbre, test[:3])
        if predite == test[3]:
            bonnes += 1
        print(f"  J{i + 1} : vraie = {test[3]}, prédite = {predite}  → {'juste' if predite == test[3] else 'FAUX'}")
    print(f"  Accuracy = {bonnes}/{len(donnees)} = {bonnes / len(donnees):.2f}")


# --- Utilisation ---
print("Construction de l'arbre ")
arbre = construire(jours, [0, 1, 2])
print()
print(" L'arbre appris ")
afficher(arbre)
print()
print("Prédictions ")
for nom, x in [("J9", ("Soleil", "Normale", "Faible")), ("J10", ("Pluie", "Haute", "Faible"))]:
    print(f"{nom} {x} → {predire(arbre, x)}")
print()
print(" Évaluation (leave-one-out) ")
evaluer(jours)

Construction de l'arbre 
gain(Ciel) = 0.033
gain(Humidité) = 0.375
gain(Vent) = 0.167
→ on choisit Humidité  (Gini 0.500, {'Oui': 4, 'Non': 4})
  branche Humidité = 65 : {'Oui': 1}
  branche Humidité = 90 : {'Non': 1}
  branche Humidité = 70 : {'Oui': 1, 'Non': 1}
    gain(Ciel) = 0.500
    gain(Vent) = 0.500
    → on choisit Ciel  (Gini 0.500, {'Oui': 1, 'Non': 1})
      branche Ciel = Pluie : {'Oui': 1}
      branche Ciel = Soleil : {'Non': 1}
  branche Humidité = 85 : {'Non': 1}
  branche Humidité = 80 : {'Non': 1}
  branche Humidité = 60 : {'Oui': 1}
  branche Humidité = 75 : {'Oui': 1}

 L'arbre appris 
SI Humidité = 65 :
    → On joue : Oui
SI Humidité = 90 :
    → On joue : Non
SI Humidité = 70 :
    SI Ciel = Pluie :
        → On joue : Oui
    SI Ciel = Soleil :
        → On joue : Non
SI Humidité = 85 :
    → On joue : Non
SI Humidité = 80 :
    → On joue : Non
SI Humidité = 60 :
    → On joue : Oui
SI Humidité = 75 :
    → On joue : Oui

Prédictions 
J9 ('Soleil', 'Normale',